In [1]:
#!pip install -U pip
#!pip install flwr==1.30.0
#!pip install ray
#!pip install "flwr[simulation]"
#!pip install torch

In [2]:
#!pip install torchvision

In [3]:
#!pip install pandas

In [1]:
import flwr
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from torchvision.transforms import Compose, Normalize, ToTensor
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Dataset

from flwr.app import ArrayRecord, Context, ConfigRecord, Message, MetricRecord, RecordDict
from flwr.clientapp import ClientApp
from flwr.serverapp import Grid, ServerApp
from flwr.serverapp.strategy import FedAvg, FedAdam, FedYogi, FedProx
from flwr.simulation import run_simulation

from functools import partial
from PIL import Image
import pandas as pd
import os

In [2]:
print(flwr.__version__)

1.30.0


In [3]:
DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)
print(DEVICE)

cuda


In [14]:
CONFIG = {
    "num-server-rounds": 10,
    "lr": 0.01,
    "model-type": "standard",      # or "bn"
    "local-epochs": 20,
    "fraction-train": 1.0,
    "strategy": "FedAvg",          # FedAvg, FedAdam, FedYogi, FedProx, GradientProjection
}

In [15]:
BASE_DIR="/home/ola/gsn"
TEST_DIR=f"{BASE_DIR}/test"
TRAIN_DIR=f"{BASE_DIR}/train"
TRAIN_LABELS=f"{BASE_DIR}/trainLabels.csv"
TEST_LABELS=f"{BASE_DIR}/testLabels.csv"

TRAIN_DATASET = None
TEST_DATASET = None

In [16]:
class RetinopathyDataset(Dataset):
    def __init__(self, image_dir, csv_file, transform=None):

        self.image_dir = image_dir
        self.transform = transform
        self.df = pd.read_csv(csv_file)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image_name = self.df.iloc[idx]["image"]
        label = self.df.iloc[idx]["level"]

        image_path = os.path.join(
            self.image_dir,
            f"{image_name}.jpeg"
        )

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

In [17]:
class Net(nn.Module):
    """Model (simple CNN adapted from 'PyTorch: A 60 Minute Blitz')"""

    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 53 * 53, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 5)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)


class NetBN(nn.Module):
    """Simple CNN with BatchNorm"""

    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 6, 5)
        self.bn1 = nn.BatchNorm2d(6)

        self.conv2 = nn.Conv2d(6, 16, 5)
        self.bn2 = nn.BatchNorm2d(16)

        self.pool = nn.MaxPool2d(2, 2)

        self.fc1 = nn.Linear(16 * 53 * 53, 120)
        self.bn3 = nn.BatchNorm1d(120)

        self.fc2 = nn.Linear(120, 84)
        self.bn4 = nn.BatchNorm1d(84)

        self.fc3 = nn.Linear(84, 5)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = x.view(x.size(0), -1)
        x = F.relu(self.bn3(self.fc1(x)))
        x = F.relu(self.bn4(self.fc2(x)))
        return self.fc3(x)


pytorch_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        (0.5, 0.5, 0.5),
        (0.5, 0.5, 0.5),
    ),
])


def load_data(partition_id, num_partitions):
    global TRAIN_DATASET, TEST_DATASET

    if TRAIN_DATASET is None:
        TRAIN_DATASET = RetinopathyDataset(
            TRAIN_DIR,
            TRAIN_LABELS,
            transform=pytorch_transforms,
        )

    if TEST_DATASET is None:
        TEST_DATASET = RetinopathyDataset(
            TEST_DIR,
            TEST_LABELS,
            transform=pytorch_transforms,
        )

    train_dataset = TRAIN_DATASET
    test_dataset = TEST_DATASET

    # deterministic partition
    total_size = len(train_dataset)
    partition_size = total_size // num_partitions
    start = partition_id * partition_size

    # last client gets remaining samples
    if partition_id == num_partitions - 1:
        end = total_size
    else:
        end = start + partition_size

    indices = list(range(start, end))

    client_dataset = torch.utils.data.Subset(
        train_dataset,
        indices
    )

    trainloader = DataLoader(
        client_dataset,
        batch_size=32,
        shuffle=True,
    )
    testloader = DataLoader(
        test_dataset,
        batch_size=32,
        shuffle=False,
    )

    return trainloader, testloader


def load_centralized_dataset():
    """Load test set and return dataloader."""

    test_dataset = RetinopathyDataset(
        image_dir=TEST_DIR,
        csv_file=TEST_LABELS,
        transform=pytorch_transforms,
    )

    return DataLoader(
        test_dataset,
        batch_size=128,
        shuffle=False,
    )


def train_local_model(net, trainloader, epochs, lr, device):
    """Train the model on the training set."""
    net.to(device)  # move model to GPU if available
    criterion = torch.nn.CrossEntropyLoss().to(device)
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    net.train()
    running_loss = 0.0
    for _ in range(epochs):
        for batch in trainloader:
            images, labels = batch
            images = images.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            loss = criterion(net(images), labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
    avg_trainloss = running_loss / len(trainloader)
    return avg_trainloss


def test(net, testloader, device):
    """Validate the model on the test set."""
    net.to(device)
    net.eval()
    criterion = torch.nn.CrossEntropyLoss()
    correct, loss = 0, 0.0
    with torch.no_grad():
        for batch in testloader:
            images, labels = batch
            images = images.to(device)
            labels = labels.to(device)
            outputs = net(images)
            loss += criterion(outputs, labels).item()
            correct += (torch.max(outputs.data, 1)[1] == labels).sum().item()
    accuracy = correct / len(testloader.dataset)
    loss = loss / len(testloader)
    return loss, accuracy

In [18]:
client_app = ClientApp()


@client_app.train()
def train(msg: Message, context: Context):
    # Build model
    model = get_model_type(CONFIG["model-type"])

    # Load global parameters
    model.load_state_dict(
        msg.content["arrays"].to_torch_state_dict(),
        strict=False
    )

    model.to(DEVICE)

    # Load client data
    partition_id = context.node_config["partition-id"]
    num_partitions = context.node_config["num-partitions"]

    trainloader, _ = load_data(
        partition_id,
        num_partitions
    )

    print(
        "CLIENT",
        context.node_config["partition-id"],
        DEVICE,
        torch.cuda.is_available()
    )

    # Train locally
    avg_loss = train_local_model(
        model,
        trainloader,
        epochs=CONFIG["local-epochs"],
        lr=CONFIG["lr"],
        device=DEVICE,
    )

    # Move parameters to CPU
    state_dict = {
        k: v.cpu()
        for k, v in model.state_dict().items()
    }

    # Return updated model
    content = RecordDict(
        {
            "arrays": ArrayRecord(state_dict),
            "metrics": MetricRecord(
                {
                    "train_loss": float(avg_loss),
                    "num-examples": len(trainloader.dataset),
                }
            ),
        }
    )

    return Message(
        content=content,
        reply_to=msg
    )


@client_app.evaluate()
def evaluate(msg: Message, context: Context):
    """Evaluate the model on local validation data."""
    print(10 * "=" + " LOCAL EVALUATE " + 10 * "=")

    # Build model
    model = get_model_type(CONFIG["model-type"])

    # Load weights
    model.load_state_dict(msg.content["arrays"].to_torch_state_dict(), strict=False)
    model.to(DEVICE)

    # Load local validation data
    partition_id = context.node_config["partition-id"]
    num_partitions = context.node_config["num-partitions"]
    _, valloader = load_data(partition_id, num_partitions)

    # Evaluate
    eval_loss, eval_acc = test(model, valloader, DEVICE)

    # Reply
    metrics = {
        "eval_loss": float(eval_loss),
        "eval_acc": float(eval_acc),
        "num-examples": len(valloader.dataset),
    }
    content = RecordDict({"metrics": MetricRecord(metrics)})
    return Message(content=content, reply_to=msg)


In [19]:
# To implement if needed
# class GradientProjection(FedAvg):

#     def __init__(
#         self,
#         lr=1e-2,
#         radius=1.0,
#         project_each_client=True,
#         project_agg=True,
#         **kwargs,
#     ):
#         super().__init__(**kwargs)

#         self.lr=lr
#         self.radius=radius
#         self.project_each_client=project_each_client
#         self.project_agg=project_agg

In [20]:
def get_strategy(name):
    if name == "FedAvg":
        return FedAvg(
            fraction_train=CONFIG["fraction-train"],
        )
    if name == "FedAdam":
        return FedAdam(
            fraction_train=CONFIG["fraction-train"],
            eta=0.001,
            eta_l=1.0,
            beta_1=0.9,
            beta_2=0.99,
            tau=1e-9,
        )
    if name == "FedYogi":
        return FedYogi(
            fraction_train=CONFIG["fraction-train"],
            eta=0.001,
            eta_l=1.0,
            beta_1=0.9,
            beta_2=0.99,
            tau=1e-3,
        )
    if name == "FedProx":
        return FedProx(
            fraction_train=CONFIG["fraction-train"],
            proximal_mu=0.01,
        )
    # if name == "GradientProjection":
    #     return GradientProjection(
    #         lr=["lr"],
    #         project_each_client=True,
    #         project_agg=True,
    #     )

    raise ValueError(f"Unknown strategy: {name}")

In [21]:
def get_model_type(name):
    if name == "bn":
        print('Using BatchNorm model')
        return NetBN()

    print('Using standard model')
    return Net()

In [22]:
server_app = ServerApp()


@server_app.main()
def main(grid: Grid, context: Context) -> None:
    """Main entry point for the ServerApp."""

    # Read run config
    num_rounds: int = CONFIG["num-server-rounds"]
    lr: float = CONFIG["lr"]
    print(f"Starting training for {num_rounds} rounds with initial lr={lr}")

    # Load global model
    global_model = get_model_type(CONFIG["model-type"])
    strategy = get_strategy(CONFIG["strategy"])

    arrays = ArrayRecord(filter_state_dict(global_model.state_dict()))

    # Start strategy, run FedAvg for `num_rounds`
    result = strategy.start(
        grid=grid,
        initial_arrays=arrays,
        train_config=ConfigRecord({"lr": lr}),
        num_rounds=num_rounds,
        evaluate_fn=partial(global_evaluate, model_type=CONFIG["model-type"]),
    )

    # Save final model to disk
    print("\nSaving final model to disk...")
    state_dict = result.arrays.to_torch_state_dict()
    torch.save(state_dict, "final_model.pt")


def global_evaluate(server_round: int, arrays: ArrayRecord, model_type: str) -> MetricRecord:
    """Evaluate model on central data."""

    print(10 * '=' + ' GLOBAL EVALUATE ' + 10 * '=')

    # Load the model and initialize it with the received weights
    model = get_model_type(CONFIG["model-type"])

    model.load_state_dict(filter_state_dict(arrays.to_torch_state_dict()), strict=False)
    model.to(DEVICE)

    # BN running stats are not aggregated across clients, so switch BN layers to
    # train mode to use batch statistics during evaluation instead of stale defaults.
    if model_type == "bn":
        for m in model.modules():
            if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d)):
                m.train()

    # Load entire test set
    test_dataloader = load_centralized_dataset()

    # Evaluate the global model on the test set
    test_loss, test_acc = test(model, test_dataloader, DEVICE)

    # Return the evaluation metrics
    return MetricRecord({"accuracy": test_acc, "loss": test_loss})


def filter_state_dict(state_dict):
    return {
        k: v for k, v in state_dict.items()
        if "num_batches_tracked" not in k
    }

In [ ]:
run_simulation(
    backend_config={
        "client_resources": {
            "num_cpus": 10,
            "num_gpus": 1,
        }
    },
    server_app=server_app,
    client_app=client_app,
    num_supernodes=3)

In [ ]:
print("Done")